In [1]:
"""
Prepare 3-hourly airport and Open-Meteo weather data and bias-correct
Open-Meteo using their common 2017-01-01 to 2023-02-28 period.

Variables retained:
    - air temperature
    - relative humidity
    - wind speed
    - shortwave radiation
    - station-level pressure

Variables deliberately excluded:
    - gust speed
    - wind direction
    - rainfall

Important timing convention
---------------------------
Hourly values are aggregated into non-overlapping, trailing 3-hour means.
For example, the row labelled 03:00 represents values in (00:00, 03:00],
i.e. the hourly records at 01:00, 02:00 and 03:00 when the source data are
regular hourly observations.
"""

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# ============================================================
# 1. SETTINGS
# ============================================================

DATA_FOLDER = Path(
    r"C:/Users/sh1345/code/myproject/Project 1/data/weather data"
)

AIRPORT_FILE = DATA_FOLDER / "airport_weather_clean_2007_to_2023_02_28.csv"
OPENMETEO_FILE = DATA_FOLDER / "open_meteo_historical_forecast_Rotorua_2017_2026.csv"

# Full, independently prepared 3-hourly weather datasets
AIRPORT_3H_FILE = DATA_FOLDER / "airport_weather_3hourly_2007_to_2023.csv"
OPENMETEO_3H_FILE = DATA_FOLDER / "openmeteo_weather_3hourly_2017_to_2026.csv"

# Comparison and bias summaries
OVERLAP_FILE = DATA_FOLDER / "airport_openmeteo_3hourly_overlap_2017_to_2023.csv"
BIAS_TABLE_FILE = DATA_FOLDER / "bias_tables_month_hour_2017_to_2023.csv"
BIAS_SUMMARY_BEFORE_FILE = DATA_FOLDER / "bias_summary_before_correction.csv"
BIAS_SUMMARY_AFTER_FILE = DATA_FOLDER / "bias_summary_after_correction.csv"

# Corrected Open-Meteo outputs
OPENMETEO_CORRECTED_3H_FILE = (
    DATA_FOLDER / "openmeteo_bias_corrected_3hourly_2017_to_2026.csv"
)
OPENMETEO_CORRECTED_FORECAST_FILE = (
    DATA_FOLDER / "openmeteo_bias_corrected_3hourly_forecast_period_2023_onward.csv"
)

DATETIME_COL = "datetime_NZ"
OPENMETEO_RAW_DATETIME_COL = "datetime (UTC+13)"

BIAS_START = pd.Timestamp("2017-01-01 00:00:00")
BIAS_END = pd.Timestamp("2023-02-28 23:59:59")

FORECAST_START = pd.Timestamp("2023-03-01 00:00:00")
FORECAST_END = None

EXPECTED_HOURLY_FREQ = pd.Timedelta(hours=1)
EXPECTED_3H_FREQ = pd.Timedelta(hours=3)

# These are the only model weather variables retained.
WEATHER_COLS = [
    "air_temperature_deg_c",
    "relative_humidity_percent",
    "wind_speed_m_s",
    "radiation_w_m2",
    "station_level_pressure_hpa",
]

# All retained variables use additive correction:
# corrected Open-Meteo = raw Open-Meteo - estimated bias
# where bias = mean(Open-Meteo - airport).
ADDITIVE_CORRECTION_COLS = WEATHER_COLS.copy()

In [2]:



# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def print_heading(title):
    print("\n" + "=" * 88)
    print(title)
    print("=" * 88)


def report_dataset(df, name, datetime_col=DATETIME_COL, expected_freq=None):
    """Print basic structure, time-step, missingness and numeric ranges."""
    print_heading(name)
    print("Shape:", df.shape)

    if df.empty:
        print("Dataset is empty.")
        return

    print("Start:", df[datetime_col].min())
    print("End:  ", df[datetime_col].max())
    print("Duplicate timestamps:", df[datetime_col].duplicated().sum())

    diffs = df[datetime_col].sort_values().diff().dropna()
    if not diffs.empty:
        print("Most common time step:", diffs.mode().iloc[0])
        if expected_freq is not None:
            print(
                f"Gaps larger than {expected_freq}:",
                int((diffs > expected_freq).sum()),
            )

    print("\nMissing values:")
    print(df.isna().sum().to_string())

    numeric_cols = [
        col for col in df.columns
        if col != datetime_col and pd.api.types.is_numeric_dtype(df[col])
    ]
    if numeric_cols:
        print("\nValue ranges:")
        for col in numeric_cols:
            print(f"{col}: min={df[col].min()}, max={df[col].max()}")


def prepare_hourly_weather(df, datetime_col, weather_cols, name):
    """
    Clean an hourly weather table before aggregation.

    Duplicate timestamps are averaged. This does not fill missing hours.
    """
    missing_cols = [col for col in weather_cols if col not in df.columns]
    if missing_cols:
        raise KeyError(f"{name} is missing required columns: {missing_cols}")

    out = df[[datetime_col] + weather_cols].copy()
    out[datetime_col] = pd.to_datetime(out[datetime_col], errors="coerce")
    out = out.dropna(subset=[datetime_col])

    for col in weather_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    duplicate_count = int(out[datetime_col].duplicated().sum())
    if duplicate_count:
        print(f"{name}: averaging {duplicate_count} duplicate timestamps.")
        out = (
            out.groupby(datetime_col, as_index=False)[weather_cols]
            .mean()
        )

    return out.sort_values(datetime_col).reset_index(drop=True)


def aggregate_to_trailing_3h(df, datetime_col, weather_cols):
    """
    Aggregate hourly weather into fixed trailing 3-hour means.

    min_count=1 retains a 3-hour bin if at least one hourly value exists.
    The accompanying count columns are used to diagnose incomplete bins.
    """
    indexed = df.set_index(datetime_col)[weather_cols].sort_index()

    values_3h = indexed.resample(
        "3h",
        label="right",
        closed="right",
        origin="start_day",
    ).mean()

    counts_3h = indexed.resample(
        "3h",
        label="right",
        closed="right",
        origin="start_day",
    ).count()

    # A normal interior bin should contain three hourly observations.
    incomplete = counts_3h.lt(3) & counts_3h.gt(0)
    empty = counts_3h.eq(0)

    print("Incomplete non-empty 3-hour bins by variable:")
    print(incomplete.sum().to_string())
    print("Completely empty 3-hour bins by variable:")
    print(empty.sum().to_string())

    return values_3h.reset_index()


def calculate_metrics_table(comparison, variables, suffix_reference, suffix_candidate):
    """Calculate airport-vs-Open-Meteo metrics for each variable."""
    rows = []

    for var in variables:
        ref_col = f"{var}{suffix_reference}"
        candidate_col = f"{var}{suffix_candidate}"

        valid = comparison[[ref_col, candidate_col]].dropna()
        if valid.empty:
            continue

        y_true = valid[ref_col].to_numpy(dtype=float)
        y_pred = valid[candidate_col].to_numpy(dtype=float)
        errors = y_pred - y_true

        rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
        mae = float(mean_absolute_error(y_true, y_pred))
        bias = float(np.mean(errors))
        median_bias = float(np.median(errors))

        try:
            r2 = float(r2_score(y_true, y_pred))
        except ValueError:
            r2 = np.nan

        mean_reference = float(np.mean(y_true))
        pbias = (
            100.0 * float(np.sum(errors)) / float(np.sum(y_true))
            if not np.isclose(np.sum(y_true), 0.0)
            else np.nan
        )

        rows.append({
            "variable": var,
            "n_overlap": len(valid),
            "airport_mean": mean_reference,
            "openmeteo_mean": float(np.mean(y_pred)),
            "mean_bias_OM_minus_airport": bias,
            "median_bias_OM_minus_airport": median_bias,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
            "PBIAS_percent": pbias,
            "airport_min": float(np.min(y_true)),
            "airport_max": float(np.max(y_true)),
            "openmeteo_min": float(np.min(y_pred)),
            "openmeteo_max": float(np.max(y_pred)),
        })

    return pd.DataFrame(rows)


def build_month_hour_bias_tables(overlap, variables, min_samples=10):
    """
    Estimate mean Open-Meteo-minus-airport bias for each month and 3-hour slot.

    Sparse month-hour groups fall back to the variable's global overlap bias.
    """
    tables = []
    global_biases = {}

    for var in variables:
        airport_col = f"{var}_airport"
        om_col = f"{var}_om"

        valid = overlap[
            ["month_NZ", "hour_NZ", airport_col, om_col]
        ].dropna().copy()

        if valid.empty:
            raise ValueError(f"No valid overlap data for {var}.")

        valid["error_OM_minus_airport"] = valid[om_col] - valid[airport_col]
        global_bias = float(valid["error_OM_minus_airport"].mean())
        global_biases[var] = global_bias

        table = (
            valid.groupby(["month_NZ", "hour_NZ"], as_index=False)
            .agg(
                estimated_bias=("error_OM_minus_airport", "mean"),
                n_samples=("error_OM_minus_airport", "size"),
            )
        )

        table["used_global_fallback"] = table["n_samples"] < min_samples
        table.loc[
            table["used_global_fallback"], "estimated_bias"
        ] = global_bias
        table["variable"] = var
        table["global_bias"] = global_bias

        tables.append(table)

    all_tables = pd.concat(tables, ignore_index=True)
    return all_tables, global_biases


def apply_month_hour_bias_correction(
    om_3h,
    bias_tables,
    global_biases,
    variables,
    datetime_col=DATETIME_COL,
):
    """Apply month-hour additive correction to the complete OM 3-hour table."""
    corrected = om_3h.copy()
    corrected["month_NZ"] = corrected[datetime_col].dt.month
    corrected["hour_NZ"] = corrected[datetime_col].dt.hour

    # Retain raw columns to make the transformation auditable.
    for var in variables:
        corrected[f"{var}_raw"] = corrected[var]

        table = bias_tables.loc[
            bias_tables["variable"].eq(var),
            ["month_NZ", "hour_NZ", "estimated_bias"],
        ].rename(columns={"estimated_bias": f"{var}_estimated_bias"})

        corrected = corrected.merge(
            table,
            on=["month_NZ", "hour_NZ"],
            how="left",
            validate="many_to_one",
        )

        bias_col = f"{var}_estimated_bias"
        corrected[bias_col] = corrected[bias_col].fillna(global_biases[var])
        corrected[var] = corrected[var] - corrected[bias_col]

    # Physically sensible bounds after correction.
    corrected["relative_humidity_percent"] = corrected[
        "relative_humidity_percent"
    ].clip(0, 100)
    corrected["wind_speed_m_s"] = corrected["wind_speed_m_s"].clip(lower=0)
    corrected["radiation_w_m2"] = corrected["radiation_w_m2"].clip(lower=0)
    corrected["station_level_pressure_hpa"] = corrected[
        "station_level_pressure_hpa"
    ].clip(800, 1100)

    output_cols = (
        [datetime_col]
        + variables
        + [f"{var}_raw" for var in variables]
        + [f"{var}_estimated_bias" for var in variables]
    )

    return corrected[output_cols].sort_values(datetime_col).reset_index(drop=True)

In [5]:



# ============================================================
# 3. READ AND STANDARDISE AIRPORT DATA
# ============================================================

airport_raw = pd.read_csv(AIRPORT_FILE)
airport_raw.columns = airport_raw.columns.str.strip()

airport_hourly = prepare_hourly_weather(
    airport_raw,
    datetime_col=DATETIME_COL,
    weather_cols=WEATHER_COLS,
    name="Airport",
)

report_dataset(
    airport_hourly,
    "CLEAN AIRPORT WEATHER BEFORE 3-HOURLY AGGREGATION",
    expected_freq=EXPECTED_HOURLY_FREQ,
)


# ============================================================
# 4. READ AND STANDARDISE OPEN-METEO DATA
# ============================================================

om_raw = pd.read_csv(OPENMETEO_FILE)
om_raw.columns = om_raw.columns.str.strip()

if OPENMETEO_RAW_DATETIME_COL not in om_raw.columns:
    raise KeyError(
        f"Open-Meteo datetime column '{OPENMETEO_RAW_DATETIME_COL}' was not found. "
        f"Available columns: {om_raw.columns.tolist()}"
    )

# This follows the convention in the original notebook. Confirm that this
# column has already been converted to New Zealand local clock time correctly,
# including daylight-saving transitions, before running the analysis.
om_raw[DATETIME_COL] = pd.to_datetime(
    om_raw[OPENMETEO_RAW_DATETIME_COL],
    errors="coerce",
    dayfirst=True,
)

om_raw = om_raw.rename(columns={
    "temperature_2m_C": "air_temperature_deg_c",
    "relative_humidity_2m_percent": "relative_humidity_percent",
    "wind_speed_10m_ms": "wind_speed_m_s",
    "shortwave_radiation_W_m2": "radiation_w_m2",
    "surface_pressure_hPa": "station_level_pressure_hpa",
})

om_hourly = prepare_hourly_weather(
    om_raw,
    datetime_col=DATETIME_COL,
    weather_cols=WEATHER_COLS,
    name="Open-Meteo",
)

report_dataset(
    om_hourly,
    "CLEAN OPEN-METEO WEATHER BEFORE 3-HOURLY AGGREGATION",
    expected_freq=EXPECTED_HOURLY_FREQ,
)



CLEAN AIRPORT WEATHER BEFORE 3-HOURLY AGGREGATION
Shape: (141659, 6)
Start: 2007-01-01 13:00:00
End:   2023-02-28 23:00:00
Duplicate timestamps: 0
Most common time step: 0 days 01:00:00
Gaps larger than 0 days 01:00:00: 0

Missing values:
datetime_NZ                   0
air_temperature_deg_c         0
relative_humidity_percent     0
wind_speed_m_s                0
radiation_w_m2                0
station_level_pressure_hpa    0

Value ranges:
air_temperature_deg_c: min=-4.9, max=30.2
relative_humidity_percent: min=22.0, max=100.0
wind_speed_m_s: min=0.0, max=18.33
radiation_w_m2: min=0.0, max=1175.0094
station_level_pressure_hpa: min=941.5, max=1006.9
Open-Meteo: averaging 10 duplicate timestamps.

CLEAN OPEN-METEO WEATHER BEFORE 3-HOURLY AGGREGATION
Shape: (82646, 6)
Start: 2017-01-01 13:00:00
End:   2026-06-07 11:00:00
Duplicate timestamps: 0
Most common time step: 0 days 01:00:00
Gaps larger than 0 days 01:00:00: 9

Missing values:
datetime_NZ                   0
air_temperature_deg

In [6]:


# ============================================================
# 5. CREATE COMPLETE, INDEPENDENT 3-HOURLY WEATHER DATASETS
# ============================================================

print_heading("AGGREGATING AIRPORT WEATHER TO TRAILING 3-HOURLY MEANS")
airport_3h = aggregate_to_trailing_3h(
    airport_hourly,
    datetime_col=DATETIME_COL,
    weather_cols=WEATHER_COLS,
)

print_heading("AGGREGATING OPEN-METEO WEATHER TO TRAILING 3-HOURLY MEANS")
om_3h = aggregate_to_trailing_3h(
    om_hourly,
    datetime_col=DATETIME_COL,
    weather_cols=WEATHER_COLS,
)

# Drop rows where every weather variable is missing. Individual variable gaps
# are retained and reported rather than silently interpolated.
airport_3h = airport_3h.dropna(subset=WEATHER_COLS, how="all").reset_index(drop=True)
om_3h = om_3h.dropna(subset=WEATHER_COLS, how="all").reset_index(drop=True)

airport_3h.to_csv(AIRPORT_3H_FILE, index=False)
om_3h.to_csv(OPENMETEO_3H_FILE, index=False)

report_dataset(
    airport_3h,
    "FULL 3-HOURLY AIRPORT WEATHER",
    expected_freq=EXPECTED_3H_FREQ,
)
report_dataset(
    om_3h,
    "FULL 3-HOURLY OPEN-METEO WEATHER",
    expected_freq=EXPECTED_3H_FREQ,
)


# ============================================================
# 6. CREATE ONLY THE 2017-2023 MATCHED OVERLAP FOR BIAS ESTIMATION
# ============================================================

comparison = pd.merge(
    airport_3h,
    om_3h,
    on=DATETIME_COL,
    how="inner",
    suffixes=("_airport", "_om"),
    validate="one_to_one",
)

overlap = comparison.loc[
    comparison[DATETIME_COL].between(BIAS_START, BIAS_END)
].copy()

if overlap.empty:
    raise ValueError(
        "No matched 3-hourly overlap was found in the requested bias period. "
        "Check datetime parsing, timezone conversion, and timestamp alignment."
    )

overlap["month_NZ"] = overlap[DATETIME_COL].dt.month
overlap["hour_NZ"] = overlap[DATETIME_COL].dt.hour

overlap.to_csv(OVERLAP_FILE, index=False)

report_dataset(
    overlap,
    "MATCHED 3-HOURLY AIRPORT–OPEN-METEO OVERLAP USED FOR BIAS ESTIMATION",
    expected_freq=EXPECTED_3H_FREQ,
)

print("\nOverlap period requested:", BIAS_START, "to", BIAS_END)
print("Overlap period obtained: ", overlap[DATETIME_COL].min(), "to", overlap[DATETIME_COL].max())
print("Unique 3-hour slots:", sorted(overlap["hour_NZ"].dropna().unique().tolist()))


# ============================================================
# 7. METRICS BEFORE CORRECTION
# ============================================================

summary_before = calculate_metrics_table(
    overlap,
    variables=WEATHER_COLS,
    suffix_reference="_airport",
    suffix_candidate="_om",
)
summary_before.to_csv(BIAS_SUMMARY_BEFORE_FILE, index=False)

print_heading("GLOBAL METRICS BEFORE BIAS CORRECTION")
print(summary_before.to_string(index=False))


# ============================================================
# 8. ESTIMATE MONTH-HOUR BIAS TABLES FROM 2017-2023 ONLY
# ============================================================

bias_tables, global_biases = build_month_hour_bias_tables(
    overlap,
    variables=ADDITIVE_CORRECTION_COLS,
    min_samples=10,
)
bias_tables.to_csv(BIAS_TABLE_FILE, index=False)

print_heading("GLOBAL FALLBACK BIASES: OPEN-METEO MINUS AIRPORT")
for variable, value in global_biases.items():
    print(f"{variable}: {value:.6f}")

print_heading("MONTH-HOUR BIAS TABLE PREVIEW")
print(bias_tables.head(30).to_string(index=False))


# ============================================================
# 9. APPLY CORRECTION TO THE FULL 2017-2026 OPEN-METEO DATASET
# ============================================================

om_corrected_3h_audit = apply_month_hour_bias_correction(
    om_3h,
    bias_tables=bias_tables,
    global_biases=global_biases,
    variables=ADDITIVE_CORRECTION_COLS,
    datetime_col=DATETIME_COL,
)

# The model-ready file contains only corrected weather variables.
om_corrected_3h = om_corrected_3h_audit[
    [DATETIME_COL] + WEATHER_COLS
].copy()

om_corrected_3h.to_csv(OPENMETEO_CORRECTED_3H_FILE, index=False)

report_dataset(
    om_corrected_3h,
    "FULL 3-HOURLY BIAS-CORRECTED OPEN-METEO WEATHER",
    expected_freq=EXPECTED_3H_FREQ,
)


# ============================================================
# 10. CHECK METRICS AFTER CORRECTION ON THE SAME OVERLAP
# ============================================================

corrected_overlap = pd.merge(
    airport_3h,
    om_corrected_3h,
    on=DATETIME_COL,
    how="inner",
    suffixes=("_airport", "_corrected"),
    validate="one_to_one",
)

corrected_overlap = corrected_overlap.loc[
    corrected_overlap[DATETIME_COL].between(BIAS_START, BIAS_END)
].copy()

summary_after = calculate_metrics_table(
    corrected_overlap,
    variables=WEATHER_COLS,
    suffix_reference="_airport",
    suffix_candidate="_corrected",
)
summary_after.to_csv(BIAS_SUMMARY_AFTER_FILE, index=False)

print_heading("GLOBAL METRICS AFTER BIAS CORRECTION")
print(summary_after.to_string(index=False))

before_after = summary_before[
    ["variable", "MAE", "RMSE", "R2", "mean_bias_OM_minus_airport"]
].merge(
    summary_after[
        ["variable", "MAE", "RMSE", "R2", "mean_bias_OM_minus_airport"]
    ],
    on="variable",
    suffixes=("_before", "_after"),
)

print_heading("BEFORE–AFTER CORRECTION COMPARISON")
print(before_after.to_string(index=False))


# ============================================================
# 11. SAVE THE FORECAST/TEST PERIOD ONLY
# ============================================================

om_forecast_3h = om_corrected_3h.loc[
    om_corrected_3h[DATETIME_COL] >= FORECAST_START
].copy()

if FORECAST_END is not None:
    om_forecast_3h = om_forecast_3h.loc[
        om_forecast_3h[DATETIME_COL] <= FORECAST_END
    ].copy()

om_forecast_3h = om_forecast_3h.sort_values(DATETIME_COL).reset_index(drop=True)
om_forecast_3h.to_csv(OPENMETEO_CORRECTED_FORECAST_FILE, index=False)

report_dataset(
    om_forecast_3h,
    "CORRECTED OPEN-METEO FORECAST/TEST PERIOD",
    expected_freq=EXPECTED_3H_FREQ,
)


# ============================================================
# 12. FINAL OUTPUT SUMMARY
# ============================================================

print_heading("SAVED FILES")
for label, path in [
    ("Full 3-hourly airport", AIRPORT_3H_FILE),
    ("Full 3-hourly Open-Meteo", OPENMETEO_3H_FILE),
    ("Matched bias overlap", OVERLAP_FILE),
    ("Month-hour bias tables", BIAS_TABLE_FILE),
    ("Metrics before correction", BIAS_SUMMARY_BEFORE_FILE),
    ("Metrics after correction", BIAS_SUMMARY_AFTER_FILE),
    ("Full corrected 3-hourly Open-Meteo", OPENMETEO_CORRECTED_3H_FILE),
    ("Corrected forecast/test Open-Meteo", OPENMETEO_CORRECTED_FORECAST_FILE),
]:
    print(f"{label}: {path}")

print("\nDone.")



AGGREGATING AIRPORT WEATHER TO TRAILING 3-HOURLY MEANS
Incomplete non-empty 3-hour bins by variable:
air_temperature_deg_c         1
relative_humidity_percent     1
wind_speed_m_s                1
radiation_w_m2                1
station_level_pressure_hpa    1
Completely empty 3-hour bins by variable:
air_temperature_deg_c         0
relative_humidity_percent     0
wind_speed_m_s                0
radiation_w_m2                0
station_level_pressure_hpa    0

AGGREGATING OPEN-METEO WEATHER TO TRAILING 3-HOURLY MEANS
Incomplete non-empty 3-hour bins by variable:
air_temperature_deg_c         10
relative_humidity_percent     10
wind_speed_m_s                10
radiation_w_m2                10
station_level_pressure_hpa    10
Completely empty 3-hour bins by variable:
air_temperature_deg_c         0
relative_humidity_percent     0
wind_speed_m_s                0
radiation_w_m2                0
station_level_pressure_hpa    0

FULL 3-HOURLY AIRPORT WEATHER
Shape: (47220, 6)
Start: 2007-01-